<a href="https://colab.research.google.com/github/kieron-s/Comp3610-Assignment3/blob/main/Comp3610_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pyspark -q

In [4]:
import time
import pandas as pd
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("COMP3610_Assignment3_NYC_Taxi")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("Warn")
print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")

Spark version: 4.0.2
Application name: COMP3610_Assignment3_NYC_Taxi


In [5]:
DATA_PATH = "/content/yellow_tripdata_2024-01.parquet"

spark_start = time.time()
df_raw = spark.read.parquet(DATA_PATH)
spark_row_count = df_raw.count()
spark_load_time = time.time() - spark_start

pandas_start = time.time()
df_pandas = pd.read_parquet(DATA_PATH)
pandas_load_time = time.time() - pandas_start

print(f"\nLoad Time Comparison")
print(f"Spark  load time : {spark_load_time:.2f}s  ({spark_row_count:,} rows)")
print(f"Pandas load time : {pandas_load_time:.2f}s  ({len(df_pandas):,} rows)")
print(f"\nInterpretation: Spark has higher overhead on a single machine due to JVM")
print(f"startup and task scheduling. However, Spark scales across clusters and handles")
print(f"datasets far exceeding available RAM, making it essential for big data workloads.")


Load Time Comparison
Spark  load time : 14.87s  (2,964,624 rows)
Pandas load time : 1.43s  (2,964,624 rows)

Interpretation: Spark has higher overhead on a single machine due to JVM
startup and task scheduling. However, Spark scales across clusters and handles
datasets far exceeding available RAM, making it essential for big data workloads.


In [6]:
print(f"DataFrame Schema")
df_raw.printSchema()
print(f"\nTotal Rows: {spark_row_count:,}")
print(f"\nPartition Information: {df_raw.rdd.getNumPartitions()}")

DataFrame Schema
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)


Total Rows: 2,964,624

Partition Information: 2


In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance"
]

initial_count = df_raw.count()
print(f"Initial Row Count: {initial_count:,}")

df_no_nulls = df_raw.dropna(subset = critical_cols)
after_null_drop = df_no_nulls.count()
print(f"After Null Drop: {after_null_drop:,} (removed {initial_count - after_null_drop:,}")

df_filtered = df_no_nulls.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 500) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

after_filter = df_filtered.count()
print(f"After Filter: {after_filter:,} (removed {after_null_drop - after_filter:,}")
print(f"Total rows removed: {initial_count - after_filter:,}")

Initial Row Count: 2,964,624
After Null Drop: 2,964,624 (removed 0
After Filter: 2,870,046 (removed 94,578
Total rows removed: 94,578


In [8]:
df_clean = df_filtered.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0)
    ).otherwise(F.lit(None).cast(DoubleType()))
).withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
).withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime")
).withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 0,
        (F.col("tip_amount") / F.col("fare_amount")) * 100
    ).otherwise(F.lit(0.0))
)

print("Derived Columns:")
df_clean.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount",
    "trip_duration_minutes", "trip_speed_mph",
    "pickup_hour", "pickup_day_of_week", "tip_percentage"
).show(5, truncate=False)

print(f"\nClean row count: {df_clean.count():,}")


Derived Columns:
+--------------------+-------------+-----------+---------------------+------------------+-----------+------------------+------------------+
|tpep_pickup_datetime|trip_distance|fare_amount|trip_duration_minutes|trip_speed_mph    |pickup_hour|pickup_day_of_week|tip_percentage    |
+--------------------+-------------+-----------+---------------------+------------------+-----------+------------------+------------------+
|2024-01-01 00:57:55 |1.72         |17.7       |19.8                 |5.212121212121212 |0          |2                 |0.0               |
|2024-01-01 00:03:00 |1.8          |10.0       |6.6                  |16.363636363636363|0          |2                 |37.5              |
|2024-01-01 00:17:06 |4.7          |23.3       |17.916666666666668   |15.739534883720932|0          |2                 |12.875536480686694|
|2024-01-01 00:36:38 |1.4          |10.0       |8.3                  |10.120481927710843|0          |2                 |20.0              |
|20

In [9]:
df_clean.createOrReplaceTempView("taxi_trips")
print("View 'taxi_trips' registered.")

View 'taxi_trips' registered.


In [15]:
print("Query 1: Top 10 Busiest Pickup Hours")
q1 = spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*)                        AS trip_count,
        ROUND(AVG(fare_amount), 2)      AS avg_fare,
        ROUND(AVG(tip_percentage), 2)   AS avg_tip_percent
    FROM taxi_trips
    GROUP BY pickup_hour
    ORDER BY trip_count DESC
    LIMIT 10
""")
q1.show()

print("Interpretation: \n\nEvening hours around 3 p.m onwards have the majority of trip volume, this is because of after work commuting.")
print("\nTip percentages tend to hover around 20% with the highest being ~2% more at night")

Query 1: Top 10 Busiest Pickup Hours
+-----------+----------+--------+---------------+
|pickup_hour|trip_count|avg_fare|avg_tip_percent|
+-----------+----------+--------+---------------+
|         18|    206281|   17.01|          22.78|
|         17|    200310|   18.12|          22.34|
|         16|    184968|   19.46|          21.83|
|         15|    184004|   19.11|           19.8|
|         19|    178810|   17.63|          22.86|
|         14|    178026|   19.27|           19.8|
|         13|    165355|   18.42|          19.78|
|         12|    159912|    17.8|          19.74|
|         21|    155910|   18.29|          21.88|
|         20|    155559|   18.05|          22.17|
+-----------+----------+--------+---------------+

Interpretation: 

Evening hours around 3 p.m onwards have the majority of trip volume, this is because of after work commuting.

Tip percentages tend to hover around 20% with the highest being ~2% more at night


In [18]:
print("Query 2: Average trip speed by day of the week")
q2 = spark.sql("""
    SELECT
        CASE pickup_day_of_week
            WHEN 1 THEN 'Sunday'    WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'   WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday'  WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
        END                                  AS day_of_week,
        ROUND(AVG(trip_speed_mph), 2)        AS avg_speed_mph,
        ROUND(AVG(trip_distance), 2)         AS avg_distance_miles,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_mins
    FROM taxi_trips
    WHERE trip_speed_mph IS NOT NULL
    GROUP BY pickup_day_of_week
    ORDER BY avg_speed_mph DESC
""")
q2.show()
print("Interpretation: \n\nThere are higher average speeds on Tuesdays and Mondays because there is less congestion and traffic which allows for higher speeds.")
print("\nIn the later end of the week, there is more traffic due to work commuting to and from, which lowers avg speed ")

Query 2: Average trip speed by day of the week
+-----------+-------------+------------------+-----------------+
|day_of_week|avg_speed_mph|avg_distance_miles|avg_duration_mins|
+-----------+-------------+------------------+-----------------+
|    Tuesday|        17.46|              4.25|            16.18|
|     Sunday|        15.97|               3.9|            14.32|
|     Monday|        13.85|              3.77|            15.85|
|     Friday|        13.41|              3.68|            15.93|
|   Saturday|        13.26|              3.39|             14.9|
|   Thursday|        12.48|              3.54|            16.43|
|  Wednesday|        12.38|              3.61|            16.26|
+-----------+-------------+------------------+-----------------+

Interpretation: 

There are higher average speeds on Tuesdays and Mondays because there is less congestion and traffic which allows for higher speeds.

In the later end of the week, there is more traffic due to work commuting to and from

In [20]:
print("Query 3: Top 5 Pickup Locations by Revenue per Day")
q3 = spark.sql("""
    WITH revenue_by_day_location AS (
        SELECT
            CASE pickup_day_of_week
                WHEN 1 THEN 'Sunday'    WHEN 2 THEN 'Monday'
                WHEN 3 THEN 'Tuesday'   WHEN 4 THEN 'Wednesday'
                WHEN 5 THEN 'Thursday'  WHEN 6 THEN 'Friday'
                WHEN 7 THEN 'Saturday'
            END                             AS day_of_week,
            pickup_day_of_week,
            PULocationID                    AS pickup_location,
            ROUND(SUM(total_amount), 2)     AS total_revenue,
            COUNT(*)                        AS trip_count
        FROM taxi_trips
        GROUP BY pickup_day_of_week, PULocationID
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (
                PARTITION BY pickup_day_of_week
                ORDER BY total_revenue DESC
            ) AS revenue_rank
        FROM revenue_by_day_location
    )
    SELECT day_of_week, pickup_location, total_revenue, trip_count, revenue_rank
    FROM ranked
    WHERE revenue_rank <= 5
    ORDER BY pickup_day_of_week, revenue_rank
""")
q3.show(35, truncate=False)
print("Interpretation: \n\nA small set of high-traffic zones (JFK, LaGuardia, Midtown)")
print("dominate revenue across all days. Airport locations rank higher on weekends.")

Query 3: Top 5 Pickup Locations by Revenue per Day
+-----------+---------------+-------------+----------+------------+
|day_of_week|pickup_location|total_revenue|trip_count|revenue_rank|
+-----------+---------------+-------------+----------+------------+
|Sunday     |132            |1564287.93   |19526     |1           |
|Sunday     |138            |763398.54    |12038     |2           |
|Sunday     |230            |346553.95    |12736     |3           |
|Sunday     |186            |264131.38    |11092     |4           |
|Sunday     |79             |263467.74    |12263     |5           |
|Monday     |132            |2054606.73   |25282     |1           |
|Monday     |138            |1021138.28   |15656     |2           |
|Monday     |161            |460145.28    |19338     |3           |
|Monday     |236            |373008.89    |18502     |4           |
|Monday     |237            |372575.48    |19214     |5           |
|Tuesday    |132            |1794987.56   |22384     |1          

In [22]:
print("Query 4: Cumulative Trip Count by Hour")
q4 = spark.sql("""
    WITH hourly_counts AS (
        SELECT pickup_hour, COUNT(*) AS trips_this_hour
        FROM taxi_trips
        GROUP BY pickup_hour
    ),
    cumulative AS (
        SELECT pickup_hour, trips_this_hour,
            SUM(trips_this_hour) OVER (
                ORDER BY pickup_hour
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cumulative_trips
        FROM hourly_counts
    )
    SELECT pickup_hour, trips_this_hour, cumulative_trips,
        ROUND(cumulative_trips * 100.0 / SUM(trips_this_hour) OVER (), 2) AS cumulative_pct
    FROM cumulative
    ORDER BY pickup_hour
""")
q4.show(24)

crossover = q4.filter(F.col("cumulative_pct") >= 50).orderBy("pickup_hour").first()
print(f"\n50% of daily trips completed by hour: {crossover['pickup_hour']}:00 ({crossover['cumulative_percent']}%)")

print("Interpretation: \n\nThe 50% threshold is crossed in the early afternoon around 3 pm, which means trip activity is concentrated to the later half of the day")


Query 4: Cumulative Trip Count by Hour
+-----------+---------------+----------------+--------------+
|pickup_hour|trips_this_hour|cumulative_trips|cumulative_pct|
+-----------+---------------+----------------+--------------+
|          0|          75247|           75247|          2.62|
|          1|          50490|          125737|          4.38|
|          2|          34976|          160713|          5.60|
|          3|          22947|          183660|          6.40|
|          4|          15284|          198944|          6.93|
|          5|          17495|          216439|          7.54|
|          6|          39415|          255854|          8.91|
|          7|          80870|          336724|         11.73|
|          8|         113506|          450230|         15.69|
|          9|         125619|          575849|         20.06|
|         10|         135425|          711274|         24.78|
|         11|         146754|          858028|         29.90|
|         12|         159912|  

In [24]:
print("Query 5: Short vs Medium vs Long trip comparison")
q5 = spark.sql("""
    SELECT
        CASE
            WHEN trip_distance < 2   THEN 'Short (<2 miles)'
            WHEN trip_distance <= 10 THEN 'Medium (2-10 miles)'
            ELSE                          'Long (>10 miles)'
        END                              AS trip_category,
        COUNT(*)                         AS trip_count,
        ROUND(AVG(fare_amount), 2)       AS avg_fare,
        ROUND(AVG(trip_distance), 2)     AS avg_distance,
        ROUND(AVG(tip_percentage), 2)    AS avg_tip_percent
    FROM taxi_trips
    GROUP BY trip_category
    ORDER BY avg_distance
""")
q5.show()

print("Interpretation: \n\nShort Trips have the most trip count volume by far as well as the smallest fare")
print("\nThe tip percentage is the highest in the short followed by long")
print("\nLong trips have the highest avg fare as well as the least amount of trips")

Query 5: Short vs Medium vs Long trip comparison
+-------------------+----------+--------+------------+---------------+
|      trip_category|trip_count|avg_fare|avg_distance|avg_tip_percent|
+-------------------+----------+--------+------------+---------------+
|   Short (<2 miles)|   1642438|    9.91|        1.13|          23.07|
|Medium (2-10 miles)|   1002534|   22.18|        3.96|          18.57|
|   Long (>10 miles)|    225074|   64.65|        21.7|          21.93|
+-------------------+----------+--------+------------+---------------+

Interpretation: 

Short Trips have the most trip count volume by far as well as the smallest fare

The tip percentage is the highest in the short followed by long

Long trips have the highest avg fare as well as the least amount of trips
